# 📅 Notebook 07: Forecasting & Analisis Deret Waktu
## Analitika Data Keuangan Sektor Publik

**Tujuan Pembelajaran:**
1. Memahami konsep deret waktu (time series)
2. Mendekomposisi tren, musiman, dan residual
3. Menerapkan Moving Average dan Exponential Smoothing
4. Membuat forecasting APBD untuk periode mendatang
5. Mengevaluasi akurasi forecast dengan MAE, RMSE, dan MAPE

---
> **Problem:** Proyeksikan perkembangan APBD pemerintah daerah untuk **3 tahun ke depan** berdasarkan data historis.

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.stattools import adfuller
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    STATSMODELS = True
    print('statsmodels berhasil dimuat ✅')
except ImportError:
    STATSMODELS = False
    print('statsmodels tidak tersedia. Menjalankan versi sederhana...')
    print('Install: pip install statsmodels')

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('Setup selesai ✅')

## 📂 2. Membuat Data Deret Waktu APBD

In [ ]:
# Dataset keuangan_pemda hanya 1 tahun, sehingga kita akan membuat
# data historis APBD multi-tahun yang realistis untuk praktik forecasting

# Buat data historis agregat APBD (simulasi data 2014-2024)
np.random.seed(42)
tahun = list(range(2014, 2025))

# Tren dasar: APBD nasional tumbuh ~8% per tahun
anggaran_base = 500  # Miliar Rp
tren = [anggaran_base * (1.08 ** i) for i in range(len(tahun))]
# Tambahkan noise realistis + efek COVID di 2020
noise = np.random.normal(0, 15, len(tahun))
covid_effect = [0, 0, 0, 0, 0, 0, -80, 10, 5, 0, 0]  # Index 6 = 2020
anggaran = [t + n + c for t, n, c in zip(tren, noise, covid_effect)]

# Realisasi: biasanya 80-92% dari anggaran
pct_realisasi = np.random.uniform(0.78, 0.93, len(tahun))
pct_realisasi[6] = 0.62  # COVID 2020: realisasi turun drastis
realisasi = [a * p for a, p in zip(anggaran, pct_realisasi)]

df_ts = pd.DataFrame({
    'Tahun': tahun,
    'Anggaran_M': [round(a, 2) for a in anggaran],
    'Realisasi_M': [round(r, 2) for r in realisasi],
    'Pct_Realisasi': [round(p * 100, 1) for p in pct_realisasi]
})
df_ts['Date'] = pd.to_datetime(df_ts['Tahun'], format='%Y')
df_ts = df_ts.set_index('Date')

print('Data Historis APBD (simulasi realistis):')
print(df_ts[['Tahun', 'Anggaran_M', 'Realisasi_M', 'Pct_Realisasi']].to_string())

## 📊 3. Visualisasi & Eksplorasi Deret Waktu

In [ ]:
# Plot deret waktu APBD
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Anggaran vs Realisasi
axes[0].fill_between(df_ts['Tahun'], df_ts['Anggaran_M'], df_ts['Realisasi_M'],
                     alpha=0.3, color='orange', label='Sisa Anggaran')
axes[0].plot(df_ts['Tahun'], df_ts['Anggaran_M'], 'b-o', linewidth=2, markersize=7, label='Anggaran APBD')
axes[0].plot(df_ts['Tahun'], df_ts['Realisasi_M'], 'g-s', linewidth=2, markersize=7, label='Realisasi APBD')
axes[0].axvspan(2020, 2020.1, color='red', alpha=0.3, label='Pandemi COVID-19')
axes[0].set_ylabel('Nilai (Miliar Rp)')
axes[0].set_title('Perkembangan Anggaran & Realisasi APBD (2014–2024)', fontweight='bold')
axes[0].legend()
axes[0].set_xticks(df_ts['Tahun'])

# Persentase Realisasi
colors = ['#e74c3c' if p < 60 else '#f39c12' if p < 80 else '#2ecc71' if p < 90 else '#27ae60'
          for p in df_ts['Pct_Realisasi']]
bars = axes[1].bar(df_ts['Tahun'], df_ts['Pct_Realisasi'], color=colors, edgecolor='white', linewidth=1)
axes[1].axhline(60, color='#e74c3c', linestyle='--', linewidth=1.5, alpha=0.7, label='60% (Cukup)')
axes[1].axhline(80, color='#f39c12', linestyle='--', linewidth=1.5, alpha=0.7, label='80% (Baik)')
axes[1].axhline(90, color='#27ae60', linestyle='--', linewidth=1.5, alpha=0.7, label='90% (Sangat Baik)')
axes[1].set_ylabel('% Realisasi')
axes[1].set_title('Persentase Realisasi APBD per Tahun', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].set_xticks(df_ts['Tahun'])
axes[1].set_ylim(0, 110)

for bar, val in zip(bars, df_ts['Pct_Realisasi']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.0f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Dekomposisi deret waktu (jika statsmodels tersedia)
if STATSMODELS:
    ts = df_ts['Anggaran_M']
    # Decompose (additive karena trend linear)
    decomp = seasonal_decompose(ts, model='additive', period=2, extrapolate_trend='freq')

    fig, axes = plt.subplots(4, 1, figsize=(12, 10))
    labels = ['Data Asli', 'Tren (Trend)', 'Musiman (Seasonal)', 'Residual/Noise']
    series = [ts, decomp.trend, decomp.seasonal, decomp.resid]
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#95a5a6']

    for ax, label, data, color in zip(axes, labels, series, colors):
        ax.plot(df_ts['Tahun'], data.values, '-o', color=color, linewidth=2, markersize=6)
        ax.set_ylabel(label, fontsize=10)
        ax.set_xticks(df_ts['Tahun'])
        if ax != axes[-1]:
            ax.tick_params(labelbottom=False)

    axes[0].set_title('Dekomposisi Deret Waktu APBD', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Dekomposisi memerlukan statsmodels. Install: pip install statsmodels')

## 📉 4. Moving Average

In [ ]:
# Moving Average (MA)
df_ts['MA_3'] = df_ts['Anggaran_M'].rolling(window=3).mean()
df_ts['MA_5'] = df_ts['Anggaran_M'].rolling(window=5).mean()

# Forecast dengan MA-3: rata-rata 3 tahun terakhir
forecast_ma3 = df_ts['Anggaran_M'].tail(3).mean()
forecast_years = [2025, 2026, 2027]
forecasts_ma = [forecast_ma3 * (1 + i * 0.05) for i in range(1, 4)]  # Asumsi tren +5% per tahun

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df_ts['Tahun'], df_ts['Anggaran_M'], 'b-o', linewidth=2, markersize=8, label='Data Aktual', zorder=5)
ax.plot(df_ts['Tahun'], df_ts['MA_3'], 'r--', linewidth=2, label='Moving Average 3-tahunan')
ax.plot(df_ts['Tahun'], df_ts['MA_5'], 'g--', linewidth=2, label='Moving Average 5-tahunan')

# Forecast
last_actual = df_ts['Anggaran_M'].iloc[-1]
ax.plot([df_ts['Tahun'].iloc[-1]] + forecast_years,
        [last_actual] + forecasts_ma,
        'r-^', linewidth=2, markersize=10, color='darkorange', label='Forecast MA-3')

# Confidence interval (sederhana)
std_noise = df_ts['Anggaran_M'].std() * 0.1
ax.fill_between(forecast_years,
                [f - std_noise * (i+1) for i, f in enumerate(forecasts_ma)],
                [f + std_noise * (i+1) for i, f in enumerate(forecasts_ma)],
                alpha=0.2, color='orange', label='Confidence Interval (±1σ)')

ax.axvline(2024.5, color='gray', linestyle=':', linewidth=1.5)
ax.text(2024.6, ax.get_ylim()[1] * 0.95, 'FORECAST →', fontsize=9, color='gray')
ax.set_xlabel('Tahun')
ax.set_ylabel('Anggaran APBD (Miliar Rp)')
ax.set_title('Forecasting APBD dengan Moving Average', fontweight='bold', fontsize=13)
ax.legend()
ax.set_xticks(list(df_ts['Tahun']) + forecast_years)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print('\nHasil Forecast (Moving Average):')
for yr, fc in zip(forecast_years, forecasts_ma):
    print(f'  {yr}: Rp {fc:.1f} Miliar')

## 🔢 5. Exponential Smoothing

In [ ]:
# Manual Exponential Smoothing untuk beberapa nilai alpha
def exp_smoothing(data, alpha):
    smoothed = [data.iloc[0]]
    for i in range(1, len(data)):
        smoothed.append(alpha * data.iloc[i] + (1 - alpha) * smoothed[-1])
    return smoothed

ts_data = df_ts['Anggaran_M']

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df_ts['Tahun'], ts_data, 'k-o', linewidth=2, markersize=8, label='Data Aktual', zorder=5)

alphas = [0.2, 0.5, 0.8]
colors = ['#3498db', '#e74c3c', '#2ecc71']
for alpha, color in zip(alphas, colors):
    smoothed = exp_smoothing(ts_data, alpha)
    ax.plot(df_ts['Tahun'], smoothed, '--', linewidth=1.8, color=color,
            label=f'α = {alpha} ({"responsif" if alpha > 0.5 else "halus"})')

ax.set_xlabel('Tahun')
ax.set_ylabel('Anggaran APBD (Miliar Rp)')
ax.set_title('Exponential Smoothing dengan Berbagai Nilai α\n(α tinggi = lebih responsif terhadap data baru)', 
             fontweight='bold')
ax.legend()
ax.set_xticks(df_ts['Tahun'])
plt.tight_layout()
plt.show()

print('\nInterpretasi:')
print('  α = 0.2 → Menghaluskan banyak (lebih mengikuti tren jangka panjang)')
print('  α = 0.5 → Keseimbangan')
print('  α = 0.8 → Sangat responsif terhadap nilai terbaru')

## 🤖 6. ARIMA Forecasting

In [ ]:
if STATSMODELS:
    from statsmodels.tsa.arima.model import ARIMA

    # Split: train (2014-2021), test (2022-2024)
    ts_train = ts_data.iloc[:8]   # 2014-2021
    ts_test  = ts_data.iloc[8:]   # 2022-2024
    years_train = df_ts['Tahun'].iloc[:8].values
    years_test  = df_ts['Tahun'].iloc[8:].values

    # ARIMA(1,1,1): AR=1, Differencing=1, MA=1
    model_arima = ARIMA(ts_train.values, order=(1, 1, 1))
    fitted_arima = model_arima.fit()

    # Prediksi test period
    forecast_test = fitted_arima.forecast(steps=len(ts_test))

    # Forecast 3 tahun ke depan
    forecast_future = fitted_arima.forecast(steps=len(ts_test) + 3)
    forecast_3yr = forecast_future[-3:]

    # Evaluasi
    mae_arima = mean_absolute_error(ts_test.values, forecast_test)
    rmse_arima = np.sqrt(mean_squared_error(ts_test.values, forecast_test))
    mape_arima = np.mean(np.abs((ts_test.values - forecast_test) / ts_test.values)) * 100

    # Plot
    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(years_train, ts_train.values, 'b-o', linewidth=2, markersize=8, label='Training Data')
    ax.plot(years_test, ts_test.values, 'g-o', linewidth=2, markersize=8, label='Test Data (Aktual)')
    ax.plot(years_test, forecast_test, 'r--^', linewidth=2, markersize=8, label='ARIMA Forecast (Test)')
    ax.plot([2025, 2026, 2027], forecast_3yr, 'm-D', linewidth=2, markersize=10,
            label='Forecast 2025-2027')
    ax.axvline(2021.5, color='gray', linestyle=':', linewidth=1.5)
    ax.set_xlabel('Tahun')
    ax.set_ylabel('Anggaran APBD (Miliar Rp)')
    ax.set_title(f'ARIMA(1,1,1) Forecasting\nMAE={mae_arima:.1f}M | RMSE={rmse_arima:.1f}M | MAPE={mape_arima:.1f}%',
                 fontweight='bold')
    ax.legend()
    ax.set_xticks(list(years_train) + list(years_test) + [2025, 2026, 2027])
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    print(f'\nHasil Forecast ARIMA(1,1,1):')
    for yr, fc in zip([2025, 2026, 2027], forecast_3yr):
        print(f'  {yr}: Rp {fc:.1f} Miliar')
    print(f'\nEvaluasi pada Test Period:')
    print(f'  MAE  : {mae_arima:.2f} Miliar Rp')
    print(f'  RMSE : {rmse_arima:.2f} Miliar Rp')
    print(f'  MAPE : {mape_arima:.2f}%')
else:
    print('ARIMA memerlukan statsmodels. Install: pip install statsmodels')

## 📊 7. Perbandingan Metode Forecasting

In [ ]:
# Ringkasan metrik evaluasi
print('PERBANDINGAN METODE FORECASTING')
print('=' * 60)
print(f'{'Metode':25} | {'MAE (M Rp)':12} | {'MAPE (%)':10} | {'Keterangan'}')
print('-' * 70)

# Moving Average — evaluasi sederhana
ma_pred = df_ts['MA_3'].dropna()
ma_actual = df_ts.loc[ma_pred.index, 'Anggaran_M']
ma_mae = mean_absolute_error(ma_actual, ma_pred)
ma_mape = np.mean(np.abs((ma_actual.values - ma_pred.values) / ma_actual.values)) * 100
print(f'{'Moving Average (MA-3)':25} | {ma_mae:10.2f} | {ma_mape:8.2f}% | Sederhana & transparan')

if STATSMODELS:
    print(f'{'ARIMA(1,1,1)':25} | {mae_arima:10.2f} | {mape_arima:8.2f}% | Statistik lengkap')

print()
print('Interpretasi MAPE:')
print('  < 10% → Sangat Baik | 10-20% → Baik | 20-50% → Wajar | > 50% → Buruk')

## 📝 8. Latihan

1. Ganti window MA dari 3 ke 2 dan 4. Bagaimana pengaruhnya terhadap MAPE?
2. Buat forecast untuk variabel `Realisasi_M` (bukan Anggaran).
3. Coba ARIMA(2,1,2). Apakah hasilnya lebih baik?
4. Hitung **Growth Rate** APBD per tahun. Apakah tren sedang naik, stabil, atau turun?
5. **Diskusi:** Dalam konteks perencanaan APBD, model mana yang lebih cocok digunakan oleh Badan Keuangan Daerah? Mengapa?

In [ ]:
# ✏️ Ruang Eksplorasi — Growth Rate APBD
df_ts['Growth_Rate'] = df_ts['Anggaran_M'].pct_change() * 100

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#27ae60' if g >= 0 else '#e74c3c' for g in df_ts['Growth_Rate'].fillna(0)]
ax.bar(df_ts['Tahun'], df_ts['Growth_Rate'].fillna(0), color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Tahun')
ax.set_ylabel('Growth Rate (%)')
ax.set_title('Pertumbuhan APBD per Tahun (%)', fontweight='bold')
ax.set_xticks(df_ts['Tahun'])

for x, y in zip(df_ts['Tahun'], df_ts['Growth_Rate'].fillna(0)):
    ax.text(x, y + (0.3 if y >= 0 else -0.8), f'{y:.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()